**Lab - 5: Semantic Segmentation ด้วย SAM และแผนที่ดาวเทียม**

หัวข้อ: การแยกวัตถุจากภาพถ่ายดาวเทียมโดยใช้ Segment Anything Model (SAM) ร่วมกับข้อมูลเชิงพื้นที่

**🎯 Learning Outcomes:**

เข้าใจการแปลง TMS เป็น GeoTIFF สำหรับพื้นที่สนใจ

ทดลองใช้งาน Segment Anything กับภาพภูมิศาสตร์

ประเมินผลเบื้องต้นของ segmentation จากภาพดาวเทียมจริง


# **🔧 สิ่งที่นักศึกษาจะได้ทำใน Lab:**

เลือกพื้นที่สนใจ (ROI) บนแผนที่โต้ตอบ

ดาวน์โหลดภาพดาวเทียม บริเวณที่เลือกเป็น GeoTIFF

เรียกใช้ Segment Anything (SAM) กับภาพที่ได้

แสดงผล Segment และวิเคราะห์ผล ว่า segmentation แม่นหรือไม่

(ขั้นสูง) ลองปรับพารามิเตอร์ของ SAM แล้วเปรียบเทียบผล





## Install dependencies

Uncomment and run the following cell to install the required dependencies.

In [ ]:
%pip install segment-geospatial

In [ ]:
import os
import leafmap
from samgeo import SamGeo, show_image, download_file, overlay_images, tms_to_geotiff

## Create an interactive map

In [ ]:
m = leafmap.Map(center=[14.07132497791134,100.80715926874633 ], zoom=16, height="800px")
m.add_basemap("SATELLITE")
m


Pan and zoom the map to select the area of interest. Use the draw tools to draw a polygon or rectangle on the map

In [ ]:
if m.user_roi_bounds() is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = [100.7619439829345, 14.07632023755061, 100.8168795942163, 14.06553866570846]

## Download a sample image

In [ ]:
image = "satellite.tif"
tms_to_geotiff(output=image, bbox=bbox, zoom=17, source="Satellite", overwrite=True)

You can also use your own image. Uncomment and run the following cell to use your own image.

In [ ]:
 #image = '/content/6348f38739ec3b000787e316.tif'

Display the downloaded image on the map.

In [ ]:
m.layers[-1].visible = False
m.add_raster(image, layer_name="Image")
m

## Initialize SAM class

Specify the file path to the model checkpoint. If it is not specified, the model will to downloaded to the working directory.

In [ ]:
sam = SamGeo(
    model_type="vit_h",
    sam_kwargs=None,
)

## Automatic mask generation

Segment the image and save the results to a GeoTIFF file. Set `unique=True` to assign a unique ID to each object.

In [ ]:

sam.generate(image, output="masks.tif", foreground=True, unique=True)


In [ ]:
sam.show_masks(cmap="binary_r")

Show the object annotations (objects with random color) on the map.

In [ ]:
sam.show_anns(axis="off", alpha=1, output="annotations.tif")

Compare images with a slider.

In [ ]:
leafmap.image_comparison(
    "satellite.tif",
    "annotations.tif",
    label1="Satellite Image",
    label2="Image Segmentation",
)

Add image to the map.

In [ ]:
m.add_raster("annotations.tif", alpha=0.5, layer_name="Masks")
m

Convert the object annotations to vector format, such as GeoPackage, Shapefile, or GeoJSON.

In [ ]:
sam.tiff_to_vector("masks.tif", "masks.gpkg")

## Automatic mask generation options

There are several tunable parameters in automatic mask generation that control how densely points are sampled and what the thresholds are for removing low quality or duplicate masks. Additionally, generation can be automatically run on crops of the image to get improved performance on smaller objects, and post-processing can remove stray pixels and holes. Here is an example configuration that samples more masks:

In [ ]:
sam_kwargs = {
    "points_per_side": 32,
    "pred_iou_thresh": 0.86,
    "stability_score_thresh": 0.92,
    "crop_n_layers": 1,
    "crop_n_points_downscale_factor": 2,
    "min_mask_region_area": 100,
}

In [ ]:
sam = SamGeo(
    model_type="vit_h",
    sam_kwargs=sam_kwargs,
)

In [ ]:
sam.generate(image, output="masks2.tif", foreground=True)

In [ ]:
sam.show_masks(cmap="binary_r")

In [ ]:
sam.show_anns(axis="off", opacity=1, output="annotations2.tif")

Compare images with a slider.

In [ ]:
leafmap.image_comparison(
    image,
    "annotations.tif",
    label1="Image",
    label2="Image Segmentation",
)

**📝 คำถามท้าย Lab (ให้นักศึกษาเขียนตอบใน cell markdown):**


1. พื้นที่ที่คุณเลือกคือบริเวณใด และมีลักษณะอย่างไร?

2. SAM สามารถแยกวัตถุใดได้ชัดเจน? วัตถุใดไม่ชัดเจน?

3. คิดว่า SAM มีข้อจำกัดอะไรในการใช้งานกับภาพถ่ายดาวเทียม?

4. คุณจะปรับปรุงผล segmentation นี้อย่างไรถ้ามีเวลามากขึ้น?
